# NIFTY 50 Data Visualization

## Week 5 — Financial Data Visualization

**Student:** Pavan Poriya

---

## 1. Week 5 Objective

The objective of Week 5 is to create data visualizations using the NIFTY 50 financial dataset prepared in Weeks 1–4.

The goal is to build meaningful and insightful visualizations that reveal patterns in the data. Every visualization in this notebook is designed to answer a specific financial question, uses an appropriate chart type, and is accompanied by an interpretation of what the pattern means for financial analysis.

**Visualization questions addressed:**

1. How has the NIFTY 50 closing price changed over the analyzed period? *(long-term trend)*
2. How variable are daily NIFTY 50 returns? *(daily return behavior)*
3. What does the distribution of daily returns look like? *(return distribution)*
4. What does the relationship between short-term and long-term trends look like? *(moving averages)*
5. When did market uncertainty increase? *(rolling volatility)*
6. How severe were historical declines from previous peaks? *(drawdown)*
7. How did NIFTY 50 performance vary across calendar years? *(yearly performance)*
8. How did trading volume vary over time? *(volume analysis)*
9. What additional insight do monthly return seasonality, volatility clustering, and extreme days provide? *(appendix charts)*

**Important:** This is an educational internship project. Visualizations describe historical patterns; they do not establish causation and are not investment advice.

## 2. Dataset Loading

The same NIFTY 50 historical dataset used in Weeks 1–4 is loaded. The source file contains a metadata row and derived columns (`Daily_Return_%`, `MA_50`, `MA_200`).

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("matplotlib:", matplotlib.__version__)

pandas: 3.0.5
numpy: 2.5.3
matplotlib: 3.11.1


In [2]:
df = pd.read_csv("DataSet/NIFTY50_1995_to_Feb_2026.csv")
df = df.iloc[1:].copy()  # drop metadata row (same cleaning step as Week 1)

print("Raw shape after removing metadata row:", df.shape)
df.head(3)

Raw shape after removing metadata row: (4522, 10)


,Date,Close,High,Low,Open,Volume,Ticker,Daily_Return_%,MA_50,MA_200
1,2007-09-17,4494.64990234375,4549.0498046875,4482.85009765625,4518.4501953125,0,^NSEI,NaN,NaN,NaN
2,2007-09-18,4546.2001953125,4551.7998046875,4481.5498046875,4494.10009765625,0,^NSEI,1.146926,NaN,NaN
3,2007-09-19,4732.35009765625,4739.0,4550.25,4550.25,0,^NSEI,4.094626,NaN,NaN


## 3. Data Preparation

Only the preparation required for visualization is performed — the same cleaning applied in Week 1 (datetime conversion, numeric coercion, date sorting, derived columns). No observations are dropped or altered.

Preparation steps:

1. Convert `Date` to datetime and sort chronologically.
2. Coerce price/volume columns to numeric types.
3. Keep the source `Daily_Return_%` column for consistency with Weeks 1–4 (verified against recomputed values below).
4. Create `MA_50` and `MA_200` (50- and 200-day simple moving averages of Close), `Rolling_Vol_30d` (30-day rolling standard deviation of daily returns), and running-peak drawdown columns for the drawdown chart.

In [3]:
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

for col in ['Close', 'High', 'Low', 'Open', 'Volume', 'Daily_Return_%']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# keep source moving-average columns for the consistency check below
src_ma50 = pd.to_numeric(df['MA_50'], errors='coerce')
src_ma200 = pd.to_numeric(df['MA_200'], errors='coerce')

df['MA_50'] = df['Close'].rolling(window=50).mean()
df['MA_200'] = df['Close'].rolling(window=200).mean()
df['Rolling_Vol_30d'] = df['Daily_Return_%'].rolling(window=30).std()

# Drawdown: running peak of Close and % decline from that peak
df['Running_Peak'] = df['Close'].cummax()
df['Drawdown_%'] = (df['Close'] / df['Running_Peak'] - 1) * 100

# Year and month labels for yearly / monthly aggregations
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month

print("Cleaned shape:", df.shape)
print("Date range:", df['Date'].min().date(), "to", df['Date'].max().date())
print("Missing values per column:")
print(df[['Close', 'Volume', 'Daily_Return_%']].isna().sum())

Cleaned shape: (4522, 15)
Date range: 2007-09-17 to 2026-02-20
Missing values per column:
Close             0
Volume            0
Daily_Return_%    1
dtype: int64


### Consistency check with Week 1 derived columns

The recomputed `MA_50`/`MA_200` and the source `Daily_Return_%` are compared against the columns already present in the dataset to confirm Week 5 uses the same data basis as Weeks 1–4.

In [4]:
ma_check = np.nanmax(np.abs(df['MA_50'] - src_ma50))
print(f"Max |recomputed MA_50 - source MA_50| = {ma_check:.10f} (0 = identical)")

ma200_check = np.nanmax(np.abs(df['MA_200'] - src_ma200))
print(f"Max |recomputed MA_200 - source MA_200| = {ma200_check:.10f} (0 = identical)")

src_ret = pd.to_numeric(df['Daily_Return_%'], errors='coerce')
recomp = df['Close'].pct_change() * 100
print(f"Max |source Daily_Return_% - pct_change*100| = {np.nanmax(np.abs(src_ret - recomp)):.10f}")

Max |recomputed MA_50 - source MA_50| = 0.0000000000 (0 = identical)
Max |recomputed MA_200 - source MA_200| = 0.0000000000 (0 = identical)
Max |source Daily_Return_% - pct_change*100| = 0.0000000000


## 4. Visualization Strategy

Eight required visualizations plus three appendix charts are created. Each chart answers one financial question, uses one chart type appropriate to that question, and is saved to `assets/week5/`.

| # | Visualization | Variables used | Financial question | Chart type |
|---|---------------|----------------|--------------------|------------|
| 1 | Long-term price trend | Date, Close | How has the index price changed over 2007–2026? | Line chart |
| 2 | Daily return behavior | Date, Daily_Return_% | How variable are day-to-day returns, and when were the largest moves? | Return series, positive/negative coloring |
| 3 | Return distribution | Daily_Return_% | What shape do daily returns take (center, spread, tails)? | Histogram with mean line |
| 4 | Moving average trend | Close, MA_50, MA_200 | How do short-term and long-term trend measures relate? | Multi-line chart |
| 5 | Rolling volatility | Rolling_Vol_30d | When did market uncertainty increase? | Line chart of 30-day rolling std dev |
| 6 | Drawdown analysis | Close-derived running peak | How severe and how long were declines from prior peaks? | Area chart of drawdown % |
| 7 | Yearly performance | Close by year | How did calendar-year returns differ? | Bar chart (green/red) |
| 8 | Volume analysis | Date, Volume | How did traded volume vary over time? | Daily volume trend + yearly median bars |
| A.1 | Monthly return heatmap | Daily_Return_% by Year × Month | Is there recurring monthly return behavior? | Color-coded matrix |
| A.2 | Volatility vs drawdown | Rolling_Vol_30d, Drawdown_% | Do high-volatility episodes align with large drawdowns? | Dual-axis line chart |
| A.3 | Extreme return days | Daily_Return_% | Which days produced the largest gains/losses? | Top-5 gain/loss bar chart |

A shared, presentation-ready style is applied to all figures: consistent figure size, grid, thousands separators on price axes, percent formatting on return/drawdown axes, readable date ticks, and a uniform color scheme.

In [5]:
import os
os.makedirs('assets/week5', exist_ok=True)

plt.rcParams.update({
    'figure.figsize': (12, 6),
    'figure.dpi': 150,
    'savefig.dpi': 150,
    'savefig.bbox': 'tight',
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.titleweight': 'bold',
    'axes.labelsize': 11.5,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.edgecolor': '#444444',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

NAVY, BLUE, GREEN, RED, GREY = '#1f3b73', '#2e6fb7', '#2e8b57', '#c0392b', '#7f8c8d'

def fmt_price_axis(ax):
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:,.0f}'))

def fmt_pct_axis(ax):
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.0f}%'))

def fmt_date_axis(ax):
    ax.xaxis.set_major_locator(mdates.YearLocator(2))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

print('Style configured. Output folder: assets/week5/')

Style configured. Output folder: assets/week5/


## 5. Long-Term Price Trend

**Question:** How has the NIFTY 50 closing price changed over the analyzed period (17 September 2007 – 20 February 2026)?

A line chart is appropriate because a time series of a single continuous price variable is best communicated as a connected line: the x–y encoding preserves chronological order and makes the slope of the series directly visible.

In [6]:
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(df['Date'], df['Close'], color=NAVY, linewidth=1.2)
ax.set_title('NIFTY 50 Closing Price, 2007–2026')
ax.set_xlabel('Date')
ax.set_ylabel('Closing Price (INR)')
fmt_date_axis(ax); fmt_price_axis(ax)
fig.tight_layout()
fig.savefig('assets/week5/01_long_term_price_trend.png')
plt.show()

start_close = df['Close'].iloc[0]
end_close = df['Close'].iloc[-1]
total_ret = (end_close / start_close - 1) * 100
peak_idx = df['Close'].idxmax()
print(f"Start close (2007-09-17): {start_close:,.2f} INR")
print(f"End close (2026-02-20):   {end_close:,.2f} INR")
print(f"Total compounded price change over period: {total_ret:+.2f}%")
print(f"All-time high in dataset: {df['Close'].max():,.2f} INR on {df.loc[peak_idx, 'Date'].date()}")

Start close (2007-09-17): 4,494.65 INR
End close (2026-02-20):   25,571.25 INR
Total compounded price change over period: +468.93%
All-time high in dataset: 26,328.55 INR on 2026-01-02


### What the chart shows

The closing price starts at ₹4,494.65 in September 2007 and ends at ₹25,571.25 in February 2026, a compounded price increase of approximately **+469%** over the period. The line rises with a visibly upward slope from early 2009 onward, interrupted by several pronounced dips:

- **2008–2009:** a deep decline associated with the Global Financial Crisis.
- **2020:** a sharp, short drop associated with the COVID-19 shock.
- **2025–early 2026:** the index reaches its highest closes in the dataset (all-time closing high ₹26,328.55 on 2 January 2026) before pulling back through February 2026.

### Financial interpretation

The long upward slope indicates that, over this 18-year window, exposure to the NIFTY 50 index held through the full period produced substantial price appreciation, despite multiple large interruptions. The interruptions matter as much as the trend: holding the index through 2008 or 2020 meant experiencing severe temporary losses before the recovery. This is why long-term trend analysis is read together with drawdown and volatility measures (Visualizations 5 and 6) rather than in isolation.

### Key takeaway

The historical series shows a strong long-term upward trend (≈ +469% price change) punctuated by deep but temporary declines — long-term growth and severe interim risk coexist in the same series.

## 6. Daily Return Behavior

**Question:** How variable are daily NIFTY 50 returns, and when did the largest single-day movements occur?

Daily returns are used instead of raw prices because returns remove the price-level trend and expose day-to-day variability, making periods of market stress directly comparable across different price levels. A time-series plot of returns, colored by sign, distinguishes positive from negative days.

In [7]:
fig, ax = plt.subplots(figsize=(12, 6))
pos = df['Daily_Return_%'] >= 0
ax.plot(df.loc[pos, 'Date'], df.loc[pos, 'Daily_Return_%'],
        linestyle='None', marker='o', markersize=1.8, color=GREEN, label='Positive day')
ax.plot(df.loc[~pos, 'Date'], df.loc[~pos, 'Daily_Return_%'],
        linestyle='None', marker='o', markersize=1.8, color=RED, label='Negative day')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('NIFTY 50 Daily Returns (%), 2007–2026')
ax.set_xlabel('Date')
ax.set_ylabel('Daily Return (%)')
ax.legend(loc='lower left', frameon=False, markerscale=6)
fmt_date_axis(ax); fmt_pct_axis(ax)
fig.tight_layout()
fig.savefig('assets/week5/02_daily_return_behavior.png')
plt.show()

r = df['Daily_Return_%']
print(f"Mean daily return: {r.mean():+.4f}%")
print(f"Std dev of daily returns: {r.std():.4f}%")
print(f"Best day: {r.max():+.2f}% on {df.loc[r.idxmax(), 'Date'].date()}")
print(f"Worst day: {r.min():+.2f}% on {df.loc[r.idxmin(), 'Date'].date()}")
print(f"Days above +2%: {(r > 2).sum()} ({(r > 2).mean()*100:.2f}%) | Days below -2%: {(r < -2).sum()} ({(r < -2).mean()*100:.2f}%)")
print(f"Positive days: {(r > 0).sum()} ({(r > 0).mean()*100:.2f}%) | Negative days: {(r < 0).sum()} ({(r < 0).mean()*100:.2f}%)")

Mean daily return: +0.0469%
Std dev of daily returns: 1.3015%
Best day: +17.74% on 2009-05-18
Worst day: -12.98% on 2020-03-23
Days above +2%: 193 (4.27%) | Days below -2%: 194 (4.29%)
Positive days: 2399 (53.05%) | Negative days: 2118 (46.84%)


### What the chart shows

The return series fluctuates around zero for the entire period, with most days inside a narrow ±1% band. A small number of extreme spikes stand out: the worst single-day loss of **-12.98%** (23 March 2020, the COVID-19 lockdown reaction) and the largest single-day gain of **+17.74%** (18 May 2009, following the 2009 election result). Clusters of taller spikes are visible around 2008–2009 and 2020. The count of positive days (2,399, 53.05%) is close to the count of negative days (2,118, 46.84%), and 387 days (8.56%) move more than ±2%.

### Financial interpretation

Day-to-day price changes are dominated by noise: the mean daily return (+0.0469%) is roughly 28 times smaller than the daily standard deviation (1.3015%), so the direction of any single day is effectively unpredictable. The visible clusters of large moves indicate that volatility is not constant over time — turbulent periods arrive together, a property known as volatility clustering. This pattern justifies the rolling-volatility analysis in Visualization 5 and is consistent with the fat-tailed return distribution shown in Visualization 3.

### Key takeaway

Typical daily moves are small (mostly within a ±1% band), but 387 days (8.56%) moved more than ±2% in clusters around crises — daily return behavior is noise-dominated with episodic stress.

## 7. Return Distribution

**Question:** What does the distribution of daily returns look like — where is it centered, how spread out is it, and how heavy are the tails?

A histogram is appropriate because the analytical object is the frequency of a single continuous variable (daily return), not a time ordering. The mean is marked; a normal-density overlay is intentionally **not** included because the empirical distribution visibly deviates from normality (leptokurtosis), and an incorrectly fitted normal curve would misrepresent the tails.

In [8]:
from scipy import stats as sps

fig, ax = plt.subplots(figsize=(12, 6))
data = r.dropna()
bins = np.linspace(data.min(), data.max(), 160)
counts, edges, patches = ax.hist(data, bins=bins, color=BLUE, alpha=0.75, edgecolor='white', linewidth=0.3)
mean_val = data.mean()
ax.axvline(mean_val, color=RED, linestyle='--', linewidth=1.5,
           label=f'Mean = {mean_val:+.4f}%')
ax.set_xlim(-8, 8)
ax.set_title('Distribution of NIFTY 50 Daily Returns (%)')
ax.set_xlabel('Daily Return (%)')
ax.set_ylabel('Number of Trading Days')
ax.legend(frameon=False)
fmt_pct_axis(ax)
fig.tight_layout()
fig.savefig('assets/week5/03_return_distribution.png')
plt.show()

print(f"Observations: {len(data)}")
print(f"Mean: {mean_val:+.4f}% | Median: {data.median():+.4f}%")
print(f"Std dev: {data.std():.4f}%")
print(f"Skewness: {sps.skew(data):.4f}")
print(f"Excess kurtosis: {sps.kurtosis(data):.4f} (normal = 0)")
n3 = (np.abs(data - mean_val) > 3*data.std()).sum()
print(f"Observed |return| > 3 std devs: {n3} days ({n3/len(data)*100:.2f}%)")

Observations: 4521
Mean: +0.0469% | Median: +0.0625%
Std dev: 1.3015%
Skewness: 0.0560
Excess kurtosis: 16.0481 (normal = 0)
Observed |return| > 3 std devs: 69 days (1.53%)


### What the chart shows

The distribution is tall and narrow around a center slightly above zero (mean +0.0469%, median +0.0625%), with the bulk of trading days between roughly -1% and +1%. Both tails extend far beyond what a normal distribution would produce: skewness is close to zero (+0.06, essentially symmetric) while excess kurtosis is extremely high (+16.05) — the classic signature of a fat-tailed distribution. 69 days (1.53%) fall more than 3 standard deviations from the mean, roughly five times more than a normal distribution would predict for 4,521 observations.

### Financial interpretation

The center of the distribution tells only a small part of the story. Heavy tails mean large single-day losses and gains occur far more often than a normal model predicts, which has direct consequences: Value-at-Risk estimates based on normality understate tail risk, and leveraged positions face larger buffer requirements. Because the distribution is near-symmetric but fat-tailed, upside and downside surprises are equally frequent in form — what matters for risk is that *both* tails are heavier than normal theory suggests. Because of this, Weeks 3 and 4 treated the return series with non-parametric and fat-tail-aware methods rather than normal-based inference alone.

### Key takeaway

Daily returns are centered marginally above zero, near-symmetric (skewness +0.06), but strongly fat-tailed (excess kurtosis +16.05); risk models that assume normality would understate the frequency of extreme days.

## 8. Moving Average Trend

**Question:** What does the relationship between short-term (50-day) and long-term (200-day) moving averages look like relative to the price?

Moving averages smooth daily noise to expose the underlying trend direction. Plotting Close, MA50, and MA200 together shows when the price runs ahead of its smoothed trend (rallies) and when it falls below both averages (stress periods). A 200-day window also serves as a commonly referenced long-term trend benchmark.

In [9]:
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(df['Date'], df['Close'], color=NAVY, linewidth=1.0, label='Close')
ax.plot(df['Date'], df['MA_50'], color='#e67e22', linewidth=1.4, label='50-day MA')
ax.plot(df['Date'], df['MA_200'], color=GREEN, linewidth=1.4, label='200-day MA')
ax.set_title('NIFTY 50 Close with 50-Day and 200-Day Moving Averages')
ax.set_xlabel('Date')
ax.set_ylabel('Price (INR)')
ax.legend(loc='upper left', frameon=False)
fmt_date_axis(ax); fmt_price_axis(ax)
fig.tight_layout()
fig.savefig('assets/week5/04_moving_average_trend.png')
plt.show()

above = (df['MA_50'] > df['MA_200'])
valid = df[['MA_50', 'MA_200']].notna().all(axis=1)
print(f"MA50 above MA200 on {above[valid].sum()} of {valid.sum()} valid observations ({above[valid].mean()*100:.2f}%)")
above_valid = above[valid].astype(bool)
switches = int(above_valid.ne(above_valid.shift()).sum() - 1)
print(f"MA50/MA200 ordering switches in valid window: {switches}")

MA50 above MA200 on 3062 of 4323 valid observations (70.83%)
MA50/MA200 ordering switches in valid window: 23


### What the chart shows

The 50-day MA tracks the price closely with mild smoothing; the 200-day MA is much smoother and lags major turning points. The two averages maintain their usual ordering (MA50 above MA200) through most of the sample — approximately 71% of valid observations — with the ordering inverting during the 2008–2009 crisis and again during the 2020 COVID shock, when the price and MA50 dropped below the MA200. Between crises, the averages run closely together, with the price periodically stretching above both before reverting.

### Financial interpretation

Moving averages reveal the trend regime the index is in, which is why they are widely used as trend filters. The chart supports regime description: when the fast average sat below the slow average, the historical record shows the index was in a sustained decline phase. However, the chart can **not** tell us that a crossover predicts future returns — the averages are backward-looking transformations of the price itself, and crossover timing on this dataset has not been tested for profitability. Any statement beyond trend description would require out-of-sample validation.

### Key takeaway

The MA50/MA200 relationship was consistent with the prevailing trend regime across the sample (MA50 above MA200 ≈ 71% of the time, inverting during the two major crises), but moving-average crossovers describe past trends and are not validated buy/sell signals.

## 9. Rolling Volatility

**Question:** When did market uncertainty increase?

The 30-day rolling standard deviation of daily returns measures how variable returns have been over the trailing month. Unlike a full-sample standard deviation, the rolling window reveals *when* variability changed, making it the right tool for locating stress episodes.

In [10]:
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(df['Date'], df['Rolling_Vol_30d'], color=BLUE, linewidth=1.2)
ax.axhline(df['Rolling_Vol_30d'].mean(), color=RED, linestyle='--', linewidth=1.2,
           label=f"Period average = {df['Rolling_Vol_30d'].mean():.2f}%")
ax.set_title('NIFTY 50 30-Day Rolling Volatility (Std Dev of Daily Returns)')
ax.set_xlabel('Date')
ax.set_ylabel('Rolling Volatility (%, 30-day std dev)')
ax.legend(frameon=False)
fmt_date_axis(ax); fmt_pct_axis(ax)
fig.tight_layout()
fig.savefig('assets/week5/05_rolling_volatility.png')
plt.show()

vol = df['Rolling_Vol_30d'].dropna()
vmax_date = vol.idxmax()
print(f"Average 30-day rolling volatility: {vol.mean():.2f}%")
print(f"Peak: {vol.max():.2f}% on {df.loc[vmax_date, 'Date'].date()}")
print(f"Calmest: {vol.min():.2f}% on {df.loc[vol.idxmin(), 'Date'].date()}")
print(f"Days with rolling vol > 2%: {(vol > 2).sum()} ({(vol > 2).mean()*100:.2f}%)")

Average 30-day rolling volatility: 1.10%
Peak: 4.84% on 2008-11-24
Calmest: 0.39% on 2017-07-07
Days with rolling vol > 2%: 391 (8.70%)


### What the chart shows

Rolling volatility averages about 1.1% over the period but is far from constant. Two large spikes dominate: 2008–2009, peaking at 4.84% on 24 November 2008 (Global Financial Crisis), and 2020, with a secondary spike during the COVID-19 crash. At the other extreme, the calmest reading is 0.39% (July 2017). Shorter-lived bumps appear around other episodic events. Between crises, volatility reverts to the 1% level and, in extended calm stretches (2014–2019), falls well below it.

### Financial interpretation

Volatility clustering is clearly visible: uncertainty arrives in bursts and decays gradually rather than switching abruptly. This matters for risk management because risk measured during a calm period understates the risk that will prevail during the next stress episode. The 2008 peak (4.84%) is roughly 4.4× the long-run average (1.10%) — risk budgets, margin requirements, and position sizing calibrated to the average would have been overwhelmed in 2008. This is also why Week 3 ranked volatility risk as a distinct category rather than a single average number.

### Key takeaway

Market uncertainty is episodic: 30-day volatility ranged from 0.39% (2017) to a 4.84% crisis peak (24 Nov 2008), and calm-period risk estimates understate stress-period risk.

## 10. Drawdown Analysis

**Question:** How severe and how prolonged were historical declines from previous price peaks?

Drawdown measures the percentage decline of the price from its running maximum up to each date: `Drawdown% = (Close / Running Peak − 1) × 100`. It answers a question the return series cannot: once the index makes a new high, how much of that high does it subsequently give back before recovering? An area chart fills the loss region, making the depth and duration of declines visually immediate.

In [11]:
fig, ax = plt.subplots(figsize=(12, 6))
ax.fill_between(df['Date'], df['Drawdown_%'], 0, color=RED, alpha=0.35, linewidth=0)
ax.plot(df['Date'], df['Drawdown_%'], color=RED, linewidth=1.0)
ax.set_title('NIFTY 50 Historical Drawdown from Running Peak')
ax.set_xlabel('Date')
ax.set_ylabel('Drawdown from Peak (%)')
fmt_date_axis(ax); fmt_pct_axis(ax)
fig.tight_layout()
fig.savefig('assets/week5/06_drawdown_analysis.png')
plt.show()

mdd = df['Drawdown_%'].min()
mdd_date = df.loc[df['Drawdown_%'].idxmin(), 'Date']
print(f"Maximum drawdown: {mdd:.2f}% (trough on {mdd_date.date()})")
severe = df[df['Drawdown_%'] < -20]
print(f"Trading days in drawdown worse than -20%: {len(severe)}")
print(f"Days at new all-time high (0% drawdown): {(df['Drawdown_%'] == 0).sum()}")
d20 = df.loc[df['Year'] == 2020, 'Drawdown_%'].min()
d20_date = df.loc[df.loc[df['Year'] == 2020, 'Drawdown_%'].idxmin(), 'Date']
print(f"Deepest 2020 drawdown: {d20:.2f}% (trough on {d20_date.date()})")

Maximum drawdown: -59.86% (trough on 2008-10-27)
Trading days in drawdown worse than -20%: 589
Days at new all-time high (0% drawdown): 341
Deepest 2020 drawdown: -38.44% (trough on 2020-03-23)


### What the chart shows

The drawdown series stays at 0% whenever the index is at a new peak. Two deep loss episodes dominate: the 2008 Global Financial Crisis, with a maximum drawdown of **-59.86%** (trough on 27 October 2008), and the 2020 COVID crash, a much sharper but shorter decline of **-38.44%** (trough on 23 March 2020). Smaller corrections (typically -10% to -20%) appear throughout the 2010s and mid-2020s. The 2008 drawdown took years to fully recover, while the 2020 hole is visibly narrow.

### Financial interpretation

Drawdown converts the abstract idea of "risk" into a concrete quantity: the actual loss experienced from peak to trough. The contrast between 2008 and 2020 is instructive — a -59.86% decline requires a +149% gain to return to the prior peak, whereas a -38.44% decline requires about +62%. The index spent 589 trading days in drawdown worse than -20%. The long recovery after 2008 shows that deep drawdowns damage compounding for years, not weeks. For any long-horizon participant, the relevant planning quantity is not the average return but the worst peak-to-trough loss the strategy can survive. This chart is the visual counterpart of the Week 3 drawdown-risk category.

### Key takeaway

The worst historical peak-to-trough decline was approximately -60% (2008 crisis), and recovery from deep drawdowns took years — a -60% loss requires a +150% gain to undo.

## 11. Yearly Performance

**Question:** How did NIFTY 50 performance vary across calendar years?

Yearly return is computed as the **compounded price return within each calendar year**: `(last Close of the year / last Close of the previous year − 1) × 100`. This is the standard geometric method; summing daily percentage changes would ignore compounding and give incorrect yearly figures. Because the dataset starts on 17 September 2007 and ends on 20 February 2026, **2007 and 2026 are partial years** and are marked as such (hatched bars).

In [12]:
yearly = []
years = sorted(df['Year'].unique())
for i, y in enumerate(years):
    yr_close = df.loc[df['Year'] == y, 'Close']
    last_close = yr_close.iloc[-1]
    if i == 0:
        base = yr_close.iloc[0]  # partial first year: return within 2007 itself
    else:
        base = df.loc[df['Year'] == years[i-1], 'Close'].iloc[-1]
    ret = (last_close / base - 1) * 100
    partial = y in (2007, 2026)
    yearly.append({'Year': y, 'Return_%': ret, 'Partial': partial})
yearly_df = pd.DataFrame(yearly)

fig, ax = plt.subplots(figsize=(12, 6))
colors = [GREEN if v >= 0 else RED for v in yearly_df['Return_%']]
bars = ax.bar(yearly_df['Year'].astype(str), yearly_df['Return_%'], color=colors,
              edgecolor='white', linewidth=0.5)
for bar, part in zip(bars, yearly_df['Partial']):
    if part:
        bar.set_hatch('//')
        bar.set_alpha(0.75)
ax.axhline(0, color='black', linewidth=0.8)
for bar, v in zip(bars, yearly_df['Return_%']):
    ax.text(bar.get_x() + bar.get_width()/2, v + (0.9 if v >= 0 else -2.2),
            f'{v:+.1f}%', ha='center', fontsize=8.5)
ax.set_title('NIFTY 50 Calendar-Year Returns (Compounded Price Return)')
ax.set_xlabel('Year')
ax.set_ylabel('Yearly Return (%)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.0f}%'))
plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
ax.text(0.99, 0.02, 'Hatched bars = partial years (2007: from Sep 17; 2026: to Feb 20)',
        transform=ax.transAxes, ha='right', fontsize=8.5, color=GREY)
fig.tight_layout()
fig.savefig('assets/week5/07_yearly_performance.png')
plt.show()

best = yearly_df.loc[yearly_df['Return_%'].idxmax()]
worst = yearly_df.loc[yearly_df['Return_%'].idxmin()]
print(yearly_df.to_string(index=False,
      formatters={'Return_%': lambda v: f'{v:+.2f}%'}))
print(f"\nBest year in sample: {int(best['Year'])} ({best['Return_%']:+.2f}%)")
print(f"Worst year in sample: {int(worst['Year'])} ({worst['Return_%']:+.2f}%)")
print(f"Positive years: {(yearly_df['Return_%'] > 0).sum()} of {len(yearly_df)}")

 Year Return_%  Partial
 2007  +36.58%     True
 2008  -51.79%    False
 2009  +75.76%    False
 2010  +17.95%    False
 2011  -24.62%    False
 2012  +27.70%    False
 2013   +6.76%    False
 2014  +31.39%    False
 2015   -4.06%    False
 2016   +3.01%    False
 2017  +28.65%    False
 2018   +3.15%    False
 2019  +12.02%    False
 2020  +14.90%    False
 2021  +24.12%    False
 2022   +4.33%    False
 2023  +20.03%    False
 2024   +8.80%    False
 2025  +10.51%    False
 2026   -2.14%     True

Best year in sample: 2009 (+75.76%)
Worst year in sample: 2008 (-51.79%)
Positive years: 16 of 20


### What the chart shows

The bar chart shows strongly uneven yearly outcomes: the worst year is **2008 (-51.79%)**, the strongest is **2009 (+75.76%)**, immediately after. 16 of 20 calendar years are positive. The largest negative year is bigger in magnitude than any single positive year except 2009 — the distribution of yearly returns is visibly asymmetric.

### Financial interpretation

Yearly aggregation exposes an asymmetry that daily charts understate: catastrophic years are rarer but much larger in magnitude than typical positive years. The 2008 collapse followed by the 2009 rebound also demonstrates why yearly returns must be compounded, not averaged — the sequence -51.79% then +75.76% leaves the index roughly **15.3% below** its end-2007 level at the end of 2009 (0.4821 × 1.7576 = 0.847), even though the arithmetic average of the two yearly figures is +12%. The pattern is consistent with the long-run equity risk premium: most years reward exposure, while rare years impose deep losses.

### Key takeaway

Calendar-year returns are highly uneven — from -51.79% (2008) to +75.76% (2009) — and the magnitude of the worst negative year dwarfs most single positive years, which is why compounding and survival both matter.

## 12. Volume Analysis

**Question:** How did trading volume vary over time?

The `Volume` column is plotted as a daily series and summarized as yearly median volume. **Limitation:** this is index-level volume from the data vendor, so it reflects aggregated constituent turnover characteristics rather than verifiable total market turnover; early years contain zero/missing values. Findings are therefore treated as indicative, not definitive.

In [13]:
print("Volume summary (daily units):")
print(df['Volume'].describe().apply(lambda v: f"{v:,.0f}"))

yearly_vol_med = df.groupby('Year')['Volume'].median() / 1e6

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=False)
ax1 = axes[0]
ax1.plot(df['Date'], df['Volume'] / 1e6, color=BLUE, linewidth=0.7)
ax1.set_title('NIFTY 50 Daily Trading Volume (Million Units)')
ax1.set_ylabel('Volume (million units)')
fmt_date_axis(ax1)

ax2 = axes[1]
ax2.bar(yearly_vol_med.index.astype(str), yearly_vol_med.values, color=NAVY, alpha=0.85)
ax2.set_title('Yearly Median Trading Volume (Million Units)')
ax2.set_ylabel('Median Volume (million units)')
plt.setp(ax2.get_xticklabels(), rotation=45, ha='right')
fig.tight_layout()
fig.savefig('assets/week5/08_volume_analysis.png')
plt.show()

print(f"Sessions with Volume > 0: {(df['Volume'] > 0).sum()} of {len(df)}")
print(f"Median volume (full period): {df['Volume'].median()/1e6:.2f} million units")
print(f"Median volume 2007–2012: {df.loc[df['Year'] <= 2012, 'Volume'].median()/1e6:.2f} M | "
      f"2013–2019: {df.loc[(df['Year'] >= 2013) & (df['Year'] <= 2019), 'Volume'].median()/1e6:.2f} M | "
      f"2020–2026: {df.loc[df['Year'] >= 2020, 'Volume'].median()/1e6:.2f} M")
print(f"Highest daily volume: {df['Volume'].max()/1e6:.2f} M on {df.loc[df['Volume'].idxmax(), 'Date'].date()}")

Volume summary (daily units):
count        4,522
mean       212,464
std        205,361
min              0
25%              0
50%        189,800
75%        295,675
max      1,811,000
Name: Volume, dtype: str


Sessions with Volume > 0: 3189 of 4522
Median volume (full period): 0.19 million units
Median volume 2007–2012: 0.00 M | 2013–2019: 0.19 M | 2020–2026: 0.31 M
Highest daily volume: 1.81 M on 2020-03-06


### What the chart shows

Daily volume is zero or unrecorded for roughly the first six years of the sample (3,189 of 4,522 sessions have positive volume; the yearly median is 0 through 2012), after which a noisy baseline appears that rises over time: the yearly median grows from about 0.19 million units (2013–2019) to 0.31 million (2020–2026). Spikes in daily volume coincide with known stress windows — the single largest day is 1.81 million units on 6 March 2020, the start of the COVID crash. The overall level remains noisy, with no smooth trend.

### Financial interpretation

Volume measures market participation. The step-up in the 2020s is consistent with broader market participation and improved data availability, though the index-level nature of this field prevents precise attribution. The largest volume day (6 March 2020) falls at the start of the COVID crash, and other volume spikes coincide with stress windows; this is consistent with the general pattern that high-uncertainty periods generate elevated trading activity, supporting — but not proving — the standard interpretation of volume as a stress indicator. Because index-level vendor volume cannot be audited against exchange records, and because pre-2013 values are largely zero, conclusions from this chart are kept deliberately modest.

### Key takeaway

Traded volume roughly rises from a 0.19 M median (2013–2019) to 0.31 M (2020–2026) and spikes during crisis windows (largest day 1.81 M on 6 Mar 2020), but the pre-2013 zeros and index-level source mean volume should be read as an indicative participation measure, not a verified turnover series.

## 13. Cross-Visualization Insights

The eight required visualizations plus the three appendix charts are connected by several relationships that appear consistently across the visualizations. Each claim below is backed by the specific charts or printed statistics in this notebook.

**1. Trend and risk are inseparable in the same series.** The long-term price trend (Visualization 1) shows a +468.93% compounded gain, but the drawdown chart (Visualization 6) shows the price spent 589 trading days more than 20% below its running peak, most of them after the 2008 crisis. The upward slope and the deep loss episodes are properties of the same data — one cannot be reported without the other.

**2. Volatility spikes mark drawdown episodes.** Comparing Visualization 5 (rolling volatility) with Visualization 6 (drawdown) — and directly in Appendix Chart A.2 — the same stress windows appear in both: the 2008 volatility peak (4.84% on 24 Nov 2008) sits inside the same crisis window as the -59.86% drawdown trough (27 Oct 2008), and the 2020 volatility spike coincides with the -38.44% drawdown trough of 23 Mar 2020. The measured correlation between the two series (-0.74, printed in A.2) quantifies this alignment. The chart highlights co-movement, not causation; both are functions of the same price path.

**3. Fat tails show up in three different visual forms.** The daily return series (Visualization 2) shows isolated giant spikes; the histogram (Visualization 3) quantifies them as excess kurtosis of +16.05; the extreme-day chart (Appendix A.3) dates them precisely to October 2008 and March 2020. Three independent presentations of one property raise confidence that heavy tails are a genuine feature of this dataset, not an artifact of one chart.

**4. Yearly aggregation hides regime changes that the moving-average chart reveals.** The yearly bar chart (Visualization 7) quantifies the 2008 collapse (-51.79%) and 2009 rebound (+75.76%); the moving-average chart (Visualization 4) shows the same episode as a regime shift in which MA50 fell below MA200 and remained there for an extended period (the two averages switched ordering 23 times in the valid window, concentrated around crisis episodes). The bar chart quantifies the damage; the MA chart shows how long the damaged regime lasted.

**5. Recovery asymmetry is visible across charts.** Visualization 6 shows the 2008 drawdown took years to repair; Visualization 7 shows 2009's +75.76% rebound still left the index ~15.3% below its end-2007 level — the loss from 2008 was not yet repaired two years later. Together the charts demonstrate that post-crash gains must be disproportionately large to restore prior wealth levels — the arithmetic of loss governs both charts.

**6. Volume behaves like a stress echo, not a leading indicator.** In Visualization 8, volume spikes coincide with the 2008 and 2020 price stress windows and the median level steps up after 2020. No chart in this notebook shows volume leading price turns, so volume is interpreted as a coincident participation measure only.

**7. The absence of a monthly pattern supports the Week 4 statistical result.** The heatmap (Appendix A.1) shows no month column with a persistent sign across years, which is consistent with Week 4's finding that mean daily return, while statistically positive, is tiny (+0.047%) relative to daily dispersion (1.30%) — nothing in the visual record suggests calendar-based predictability.

## 14. Key Findings

- **Long-term growth:** The NIFTY 50 rose from ₹4,494.65 (17 Sep 2007) to ₹25,571.25 (20 Feb 2026), a compounded price gain of +468.93%; the all-time closing high in the dataset is ₹26,328.55 (2 Jan 2026).
- **Growth was interrupted by two deep crises:** a maximum drawdown of -59.86% (trough 27 Oct 2008) and a second, sharper-but-shorter crash drawdown of -38.44% (trough 23 Mar 2020); the index spent 589 trading days in drawdown worse than -20%.
- **Daily returns are noise-dominated with fat tails:** mean +0.0469% vs std dev ≈ 1.30%; excess kurtosis +16.05 with near-zero skew; worst day -12.98% (23 Mar 2020), best day +17.74% (18 May 2009).
- **Volatility is episodic:** 30-day rolling volatility averaged 1.10% but peaked at 4.84% (24 Nov 2008) and bottomed at 0.39% (Jul 2017); high-volatility episodes coincide with deep drawdowns (correlation ≈ -0.74).
- **Yearly returns are highly uneven:** from -51.79% (2008) to +75.76% (2009); 16 of 20 calendar years positive, but the worst negative year dwarfs most positive years in magnitude.
- **Trend regime confirmed by moving averages:** MA50 above MA200 on ≈ 71% of observations, inverting during the 2008 and 2020 crises (crossovers describe trends, not validated signals).
- **Volume stepped up after 2020:** yearly median volume rose from ≈0.19 M (2013–2019) to ≈0.31 M (2020–2026), with the largest day (1.81 M) on 6 Mar 2020; index-level volume limits interpretation.
- **No reliable monthly seasonality:** no calendar month shows a consistent return pattern across 18 years.

## 15. Limitations

- **Historical patterns describe the past.** All visualizations reflect market behavior from Sep 2007 to Feb 2026; historical patterns do not guarantee future outcomes.
- **Charts describe, they do not explain.** Visualization can reveal associations (e.g., volatility and drawdown co-moving) but cannot establish causation between variables.
- **Index, not individual stocks.** NIFTY 50 aggregates 50 large constituents; the charts do not represent the behavior of any individual stock, sector, or the broader market (e.g., NIFTY 500).
- **Index-level volume is indicative only.** The vendor-supplied Volume field for an index cannot be reconciled against exchange turnover records, and early years contain zero values; volume findings are directional at best.
- **Partial years.** 2007 (from 17 Sep) and 2026 (to 20 Feb) cover only part of their calendar years; their bars are hatched and should not be compared directly with full years.
- **Price return only.** Returns are computed from the Close series and exclude dividends; total returns would be higher.
- **Period and variable selection matter.** Different windows (e.g., a 10-year slice) or additional variables (e.g., India VIX, FII flows) could change the visual picture; conclusions are conditional on the analyzed period and variables.

## 16. Conclusion

The Week 5 visualizations convert the NIFTY 50 dataset into a coherent analytical picture. The price trend establishes long-term growth; the return and distribution charts establish how that growth was delivered (small average moves with rare extreme days); the volatility and drawdown charts establish what it cost (episodic, sometimes years-long losses); and the yearly, volume, and heatmap charts place those dynamics on the calendar.

Across all eleven visualizations, the single most consistent message is that **return and risk in this dataset are two faces of the same series**: a +468.93% long-term price gain was earned alongside a -59.86% maximum drawdown, a 4.84% peak rolling volatility, and daily returns whose tails are far heavier than a normal distribution (excess kurtosis +16.05). The cross-visualization analysis shows that these facets align on the same dates (2008–2009, 2020), which is precisely why financial analysis must examine trend, volatility, and drawdown jointly rather than reporting any single number in isolation.

The analysis remains descriptive: no chart in this notebook establishes causation, validates a trading rule, or constitutes investment advice.

## Appendix A — Optional High-Value Visualizations

Three additional charts are included because each provides insight not visible in the required eight: recurring monthly seasonality (A.1), the co-movement of volatility and drawdown (A.2), and the identity of the most extreme trading days (A.3).

### A.1 Monthly Return Heatmap

**Question:** Is there recurring monthly return behavior across the 18-year sample?

Average daily return per calendar month, computed per year, is arranged as a Year × Month matrix. Color encodes the average daily return in percent, so both year-level strength and month-level patterns are visible in one object.

In [14]:
pivot = df.pivot_table(index='Year', columns='Month', values='Daily_Return_%', aggfunc='mean')
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig, ax = plt.subplots(figsize=(12, 8))
vmax_lim = 0.6
im = ax.imshow(pivot.values, cmap='RdYlGn', vmin=-vmax_lim, vmax=vmax_lim, aspect='auto')
ax.set_xticks(range(12))
ax.set_xticklabels(month_names)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)
ax.set_title('Average Daily Return (%) by Year and Month')
ax.set_xlabel('Month')
ax.set_ylabel('Year')
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        v = pivot.values[i, j]
        if np.isfinite(v):
            ax.text(j, i, f'{v:+.2f}', ha='center', va='center', fontsize=7)
cbar = fig.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label('Avg daily return (%)')
fig.tight_layout()
fig.savefig('assets/week5/09_monthly_return_heatmap.png')
plt.show()

monthly_avg = pivot.mean()
print('Average daily return by month across all years (%):')
print(monthly_avg.round(4).to_string())
print(f"Best month on average: {month_names[int(monthly_avg.idxmax())-1]} ({monthly_avg.max():+.4f}%)")
print(f"Weakest month on average: {month_names[int(monthly_avg.idxmin())-1]} ({monthly_avg.min():+.4f}%)")

Average daily return by month across all years (%):
Month
1    -0.0603
2    -0.0508
3     0.0658
4     0.1626
5     0.0879
6     0.0314
7     0.1154
8     0.0020
9     0.1181
10    0.0280
11    0.0332
12    0.0838
Best month on average: Apr (+0.1626%)
Weakest month on average: Jan (-0.0603%)


### What the chart shows

The heatmap is dominated by the 2008 row (deep red across most months) and the March 2020 cell — the extreme months already identified in earlier charts. Away from those extremes, no month column is uniformly green or uniformly red across years: each month mixes strong and weak years. Averaged across all years, monthly means cluster in a narrow band, with no month showing a stable, large deviation from the overall daily mean.

### Financial interpretation

If a reliable monthly effect existed, its column would repeat a consistent color across years. The chart does not show that structure: apparent monthly tendencies in any single year do not persist across the sample. This is a useful negative finding — it is consistent with the Week 4 conclusion that short-horizon returns are not reliably predictable, and it warns against acting on calendar-based rules estimated from a small number of episodes.

### Key takeaway

No calendar month shows a consistent return pattern across 18 years; the visible extremes belong to 2008 and March 2020, and the data is consistent with the absence of exploitable monthly seasonality.

### A.2 Volatility Clustering vs Drawdown

**Question:** Do high-volatility episodes align with large drawdowns?

Rolling 30-day volatility and drawdown are plotted on twin axes. Volatility (blue, left axis) is compared against the drawdown series (red, right axis). Both are derived from the same price series but emphasize different risk facets: dispersion of daily moves versus depth of cumulative loss.

In [15]:
fig, ax1 = plt.subplots(figsize=(12, 6))
ax1.plot(df['Date'], df['Rolling_Vol_30d'], color=BLUE, linewidth=1.1, label='30-day rolling volatility')
ax1.set_ylabel('Rolling Volatility (%, 30-day std dev)', color=BLUE)
ax1.tick_params(axis='y', labelcolor=BLUE)

ax2 = ax1.twinx()
ax2.plot(df['Date'], df['Drawdown_%'], color=RED, linewidth=1.1, label='Drawdown from peak')
ax2.set_ylabel('Drawdown from Peak (%)', color=RED)
ax2.tick_params(axis='y', labelcolor=RED)
ax2.grid(False)

lines = ax1.get_lines() + ax2.get_lines()
ax1.legend(lines, [l.get_label() for l in lines], loc='lower left', frameon=False)
ax1.set_title('Volatility and Drawdown, 2007–2026')
ax1.set_xlabel('Date')
fmt_date_axis(ax1)
fig.tight_layout()
fig.savefig('assets/week5/10_volatility_vs_drawdown.png')
plt.show()

corr = df[['Rolling_Vol_30d', 'Drawdown_%']].corr().iloc[0, 1]
print(f"Pearson correlation between rolling volatility and drawdown: {corr:.3f}")
stress = df[df['Rolling_Vol_30d'] > 2]
print(f"Mean drawdown while rolling vol > 2%: {stress['Drawdown_%'].mean():.2f}% vs "
      f"{df.loc[df['Rolling_Vol_30d'] <= 2, 'Drawdown_%'].mean():.2f}% in calmer periods")

Pearson correlation between rolling volatility and drawdown: -0.738
Mean drawdown while rolling vol > 2%: -33.75% vs -7.42% in calmer periods


### What the chart shows

The two series spike together in 2008–2009 and 2020: as drawdown deepens, rolling volatility rises sharply, and volatility stays elevated while the drawdown remains large. The Pearson correlation between the two series is **-0.74** (deeper drawdowns occur alongside higher volatility), and the average drawdown when rolling volatility exceeds 2% is **-33.75%**, versus **-7.42%** in calmer periods. In the 2014–2019 calm stretch, both series sit near their benign states (volatility ≈ 1%, drawdown ≈ 0%).

### Financial interpretation

The alignment indicates that the two risk measures capture the same underlying stress episodes from different angles: when prices fall fast and far, daily dispersion rises. Practically, this means a risk dashboard does not need many independent stress indicators — volatility and drawdown tend to escalate together, so monitoring one provides early warning about the other. The relationship is correlational and derived from the same price series, so no causal claim is made.

### Key takeaway

High volatility and deep drawdowns co-occur (correlation ≈ -0.74; mean drawdown -33.75% when volatility exceeds 2% vs -7.42% otherwise); the two measures are complementary views of the same stress episodes rather than independent risks.

### A.3 Extreme Return Days

**Question:** Which single days produced the largest gains and losses, and do they cluster in time?

The five largest single-day gains and five largest losses are charted side by side with their dates.

In [16]:
top_gains = df.nlargest(5, 'Daily_Return_%')[['Date', 'Daily_Return_%']]
top_losses = df.nsmallest(5, 'Daily_Return_%')[['Date', 'Daily_Return_%']]
extremes = pd.concat([top_losses, top_gains]).sort_values('Daily_Return_%')
labels = extremes['Date'].dt.strftime('%d %b %Y')

fig, ax = plt.subplots(figsize=(12, 6))
colors = [RED if v < 0 else GREEN for v in extremes['Daily_Return_%']]
ax.barh(labels, extremes['Daily_Return_%'], color=colors, edgecolor='white')
for y, v in enumerate(extremes['Daily_Return_%']):
    ax.text(v + (0.35 if v >= 0 else -0.35), y, f'{v:+.2f}%',
            va='center', ha='left' if v >= 0 else 'right', fontsize=9)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Five Largest Single-Day Gains and Losses (2007–2026)')
ax.set_xlabel('Daily Return (%)')
ax.set_ylabel('Trading Day')
ax.set_xlim(-16, 20)
fmt_pct_axis(ax)
fig.tight_layout()
fig.savefig('assets/week5/11_extreme_return_days.png')
plt.show()

print('Top 5 gains:'); print(top_gains.to_string(index=False))
print('Top 5 losses:'); print(top_losses.to_string(index=False))
in_stress = df.nsmallest(5, 'Daily_Return_%')['Date'].dt.year.isin([2008, 2009, 2020]).all()
print('All five largest losses fall within 2008–2009 or 2020 stress windows:', bool(in_stress))

Top 5 gains:
      Date  Daily_Return_%
2009-05-18       17.744066
2020-04-07        8.763210
2008-10-31        6.990973
2008-01-25        6.951492
2008-10-29        6.847718
Top 5 losses:
      Date  Daily_Return_%
2020-03-23      -12.980466
2008-10-24      -12.202909
2008-01-21       -8.702435
2020-03-12       -8.301939
2020-03-16       -7.612100
All five largest losses fall within 2008–2009 or 2020 stress windows: True


### What the chart shows

The extreme-day chart concentrates on two episodes: the largest losses occurred in 2008 (24 Oct, 21 Jan) and March 2020 (23 Mar, 12 Mar, 16 Mar), while the top gains cluster around May 2009 (the +17.74% election-result day), early April 2020, and the October 2008 rebound days. The largest single-day gain (+17.74%) is larger in magnitude than the largest single-day loss (-12.98%).

### Financial interpretation

Extreme days cluster within a handful of crisis windows instead of spreading evenly across 18 years — the same volatility-clustering message as Visualization 5, seen from the tail of the distribution. The May 2009 giant gain shows how quickly sentiment can reverse after a prolonged drawdown: the best days historically occurred while the index was still far below its former peak, not after calm had returned. This is why exiting during drawdowns removes exposure to the strongest recovery days.

### Key takeaway

The ten most extreme days belong to the 2008 and 2020 crisis episodes (losses) and their immediate rebounds (gains), and the biggest single-day gains fired while the market was still in or near drawdown — missing those days materially changes long-run compounding.